# 2 - Limits, and every comparison

Everything downstream of Combine. Reads `limits.csv` files and draws the figures
used in the slides.

The organising idea: **every comparison function takes `{label: rows}`**, so the
same call draws one campaign, five methods, or two planes depending only on what
you hand it. Nothing below is specific to a method or a campaign.

Command-line equivalent:

```bash
python sidm/scripts/abcd_plots.py results --limits <campaign>/limits_...
python sidm/scripts/abcd_plots.py compare --reference counting label=path ...
```

In [ ]:
import sys
from pathlib import Path

STUDY = Path.cwd()
REPO = STUDY.parents[2]
for p in (str(REPO), str(STUDY)):
    if p not in sys.path:
        sys.path.insert(0, p)

import plot_tools as pt

CAMPAIGN = "sixd_inclusive_v1"
CAMP = STUDY / "campaigns" / CAMPAIGN
PLOTS = STUDY / "plots" / CAMPAIGN
theory = pt.theory_by_mass()
print("model cross sections [fb]:", {int(k): round(v, 4) for k, v in theory.items()})

## One result set: the single-campaign suite

Brazil bands across the grid, limit against bound state mass with the model
cross section overlaid, the exclusion map, and the kinetic mixing plane.

On an `Lxy` axis the model prediction is a **horizontal line** because the cross
sections depend only on `m_bound`; on a mass axis the same four numbers become a
**curve**. No interpolation either way.

In [ ]:
rows = pt.as_rows(CAMP / "limits_6d_shape_iso_dphi" / "limits.csv")
print(len(rows), "points")

for path in pt.result_set(rows, PLOTS / "results", CAMPAIGN, theory):
    print(" ", path.name)

## Several methods on shared axes

`load_sets` skips anything missing, so an incomplete set of runs still plots.
The labels are what appear in the legends.

In [ ]:
sets = pt.load_sets({
    "counting":                 CAMP / "limits_6d_counting_iso_iso",
    r"ABCD iso$\times$iso":      CAMP / "limits_6d_abcd_iso_iso",
    r"ABCD iso$\times|\Delta\phi|$": CAMP / "limits_6d_abcd_iso_dphi",
    r"shape $m_{LJLJ}$ iso$\times$iso":      CAMP / "limits_6d_shape_iso_iso",
    r"shape $m_{LJLJ}$ iso$\times|\Delta\phi|$": CAMP / "limits_6d_shape_iso_dphi",
})
pt.summary_table(sets)

In [ ]:
OUT = PLOTS / "comparison"

for fs in ("2Mu2E", "4Mu"):
    pt.compare_vs_lxy(sets, fs, OUT, CAMPAIGN, theory, name=f"methods_vs_lxy_{fs}")
pt.compare_vs_mass(sets, OUT, CAMPAIGN, theory, name="methods_vs_mass")
pt.compare_exclusion(sets, OUT, CAMPAIGN, name="methods_exclusion")
print("done")

## Ratio to a reference

`against="m_bound"` is how to ask whether a method gains more at high mass - it
should, since a heavier bound state pushes the signal to higher `m_ljlj`, away
from a steeply falling background.

`against="lxy"` should be roughly flat: the background is the same at every
lifetime and only the signal acceptance moves, so it cancels in the ratio.

In [ ]:
pt.compare_ratio(sets, "counting", OUT, CAMPAIGN, against="lxy",
                 name="methods_ratio_vs_lxy")
pt.compare_ratio(sets, "counting", OUT, CAMPAIGN, against="m_bound",
                 name="methods_ratio_vs_mass")
print("done")

## Across campaigns

Same functions, different sets. These differ in selection as well as method, so
label them as such rather than reading the difference as a method effect.

In [ ]:
CAMPAIGNS = STUDY / "campaigns"
across = pt.load_sets({
    "counting (golden)":  CAMPAIGNS / "golden_hotspot_iso025_v1" / "limits",
    "ABCD (golden)":      CAMPAIGNS / "golden_hotspot_iso025_v1" / "limits_abcd",
    "ABCD (muIso)":       CAMPAIGNS / "muIso_dPhi_eHEM_skimv1_v1" / "limits_abcd_dphi",
    "shape (6D)":         CAMP / "limits_6d_shape_iso_dphi",
})
pt.summary_table(across)
for fs in ("2Mu2E", "4Mu"):
    pt.compare_vs_lxy(across, fs, STUDY / "plots" / "campaigns", "across campaigns",
                      theory, name=f"campaigns_vs_lxy_{fs}")
print("done")